# Lab - Your First Claude Request

Three short tasks. By the end you will have sent a real Claude request, learned how
to control the length of a reply, and looked inside the object the API sends back.

| Task | What you learn |
| --- | --- |
| 1 | Send a request and read the reply |
| 2 | `stop_reason` - why Claude stopped writing |
| 3 | What else comes back besides the text |

**You do not need an API key.** This lab ships with a Claude simulator, so every cell
runs offline. The code you write is exactly the code you would write against the real
API - set `ANTHROPIC_API_KEY` at home and the same notebook calls Claude for real.

**Every task tells you where to look.** Each cell names the moment in the lecture
that walks through the same code, and gives you a template to fill in. You are never
expected to invent anything from memory.

## Setup

Run this cell first, before anything else. Click it, then press **Shift+Enter**.

It is the only cell in this lab that differs from a real project: the first two lines
load the simulator. Everything below them is ordinary Claude API code.

In [ ]:
# --- Lab setup (provided - just run it) ---
import shopassist_lab
from shopassist_lab import check

# Everything below this line is ordinary Claude API code.
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()

model = "claude-sonnet-4-6"

RETURN_REQUEST = "A customer wants to return an order. Write a short helpful response."

---

## Task 1 - Make your first Claude request

Read the comment block in the cell below. It explains what to do, why it matters,
where the lecture shows the same code, and gives you a template to copy.

In [ ]:
# ============================================================
# TASK 1 - Make your first Claude request
# ============================================================
#
# WHAT TO DO
#   Send one message to Claude and print the reply.
#
# WHY IT MATTERS
#   Every Claude application is built on this one call. Tools,
#   agents and the agentic loop later in this course are all
#   this same call, repeated.
#
# WHERE TO SEE IT IN THE LECTURE
#   "Now let's make the first request" - about 3 minutes 43
#   seconds in. The video types this exact code.
#
# HOW TO DO IT
#   The code is already written below. Replace each ... with
#   the value named in the comment next to it.
#
#   Note that you assign the WHOLE result to `message`, not
#   message.content[0].text - Task 3 needs the full object.
# ============================================================

# BEGIN SOLUTION
message = client.messages.create(
    model=model,
    max_tokens=300,
    messages=[
        {
            "role": "user",
            "content": RETURN_REQUEST,
        }
    ],
)
# SCAFFOLD: message = client.messages.create(
# SCAFFOLD:     model=...,        # the model variable set up above
# SCAFFOLD:     max_tokens=...,   # use 300 - plenty for a short reply
# SCAFFOLD:     messages=[
# SCAFFOLD:         {
# SCAFFOLD:             "role": "user",
# SCAFFOLD:             "content": ...,   # the RETURN_REQUEST variable
# SCAFFOLD:         }
# SCAFFOLD:     ],
# SCAFFOLD: )
# END SOLUTION: replace each ... below with the value named beside it

print(message.content[0].text)

check("first_request", message=message)

---

## Task 2 - Read `stop_reason` and control response length

In [ ]:
# ============================================================
# TASK 2 - Read stop_reason and control response length
# ============================================================
#
# WHAT TO DO
#   Send the same prompt twice, with a different token budget
#   each time, and compare why Claude stopped writing.
#
# WHY IT MATTERS
#   Claude always tells you why it stopped. Your application
#   has to react differently to each answer: "end_turn" means
#   the reply is complete and safe to show a customer,
#   "max_tokens" means it was cut off mid-sentence. Later in
#   this course the agentic loop runs on exactly this signal.
#
# WHERE TO SEE IT IN THE LECTURE
#   "Print message stop reason" - about 4 minutes 56 seconds in.
#
# HOW TO DO IT
#   Both calls are written below, identical except for one
#   number. Fill in the two token budgets.
#
# WHAT YOU SHOULD SEE
#   max_tokens for the short one, end_turn for the long one.
#   The short reply stops mid-sentence: you did not hit a
#   content limit, you ran out of budget.
# ============================================================

# BEGIN SOLUTION
short_message = client.messages.create(
    model=model,
    max_tokens=20,
    messages=[{"role": "user", "content": RETURN_REQUEST}],
)

long_message = client.messages.create(
    model=model,
    max_tokens=300,
    messages=[{"role": "user", "content": RETURN_REQUEST}],
)
# SCAFFOLD: short_message = client.messages.create(
# SCAFFOLD:     model=model,
# SCAFFOLD:     max_tokens=...,   # use 20 - deliberately too small
# SCAFFOLD:     messages=[{"role": "user", "content": RETURN_REQUEST}],
# SCAFFOLD: )
# SCAFFOLD:
# SCAFFOLD: long_message = client.messages.create(
# SCAFFOLD:     model=model,
# SCAFFOLD:     max_tokens=...,   # use 300 - room to finish
# SCAFFOLD:     messages=[{"role": "user", "content": RETURN_REQUEST}],
# SCAFFOLD: )
# END SOLUTION: replace each ... below with the number named beside it

print("short stop_reason:", short_message.stop_reason)
print("long  stop_reason:", long_message.stop_reason)
print()
print("The truncated reply ends mid-sentence:")
print(repr(short_message.content[0].text))

check("stop_reason", short=short_message, long=long_message)

---

## Task 3 - Inspect the full response object

In [ ]:
# ============================================================
# TASK 3 - Inspect the full response object
# ============================================================
#
# WHAT TO DO
#   Print the whole `message` from Task 1, then pull three
#   values out of it.
#
# WHY IT MATTERS
#   The reply text is one field among many. `usage` is how you
#   track what a conversation costs. `id` is what you write to
#   your logs when a customer complains. `stop_reason` is what
#   your code branches on. A real application reads all of
#   these, not just the text.
#
# WHERE TO SEE IT IN THE LECTURE
#   "Finally, let's print the full response object" - about
#   5 minutes 44 seconds in.
#
# HOW TO DO IT
#   The first line below is done for you as an example: it
#   reaches into the object with a dot. Finish the other two
#   the same way. The fields you need are called input_tokens
#   and output_tokens, and both live on message.usage.
#
#   Do not retype the numbers you see printed. The point is to
#   reach into the object the way your backend would.
# ============================================================

print(message)
print()

# BEGIN SOLUTION
reply_text = message.content[0].text
input_tokens = message.usage.input_tokens
output_tokens = message.usage.output_tokens
# SCAFFOLD: reply_text = message.content[0].text
# SCAFFOLD: input_tokens = ...     # message.usage has a field called input_tokens
# SCAFFOLD: output_tokens = ...    # and one called output_tokens
# END SOLUTION: finish the two lines below by naming the field you need

print("id:           ", message.id)
print("model:        ", message.model)
print("role:         ", message.role)
print("stop_reason:  ", message.stop_reason)
print("input tokens: ", input_tokens)
print("output tokens:", output_tokens)

check("response_object", message=message, reply_text=reply_text,
      input_tokens=input_tokens, output_tokens=output_tokens)

---

## Done

You have made a Claude request, controlled how long the reply can be, learned what
`stop_reason` tells your application, and looked inside the response object.

Two commands worth knowing if you want to poke around:

| Call | What it does |
| --- | --- |
| `lab_info()` | what the simulator does and does not reproduce |
| `explain(message)` | why the simulator returned that particular reply |